# Z.AI total usages OCR

This notebook mirrors `zai_receipt_ocr.ipynb`, but it reads every file from `images/total_usages`, runs `client.layout_parsing.create(...)` first, then sends the parsed markdown to `client.chat.completions.create(...)` for structured extraction. It reports token usage, computes cost, and returns structured JSON with `total_usages` and `usage_unit` so electricity, water, and other unit variations can be handled.


In [28]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.parse import quote

from IPython.display import IFrame, Image, display
from zai import ZaiClient


In [30]:
def load_env_file(env_path: Path = Path('.env')) -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip("\"").strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def build_extraction_prompt(layout_markdown: str) -> str:
    return f'''Extract the utility bill fields from the layout parsing markdown below.
Return only valid JSON in this shape:
{{
  "provider_name": "",
  "statement_date": "",
  "usage_unit": "",
  "total_usages": null
}}

Rules:
- `total_usages` means the current month or current billing period total usage only.
- Do not return previous month usage, year-to-date usage, average usage, meter reading, or cumulative usage.
- `total_usages` must be numeric only.
- `usage_unit` must contain the usage unit exactly as shown on the bill (for example `kWh`, `L`, `m3`, or similar).
- If the bill has multiple usage values, choose the final total billed usage for the current billing period.

Layout parsing markdown:
{layout_markdown}
'''.strip()


def parse_json_content(content: object) -> dict[str, object]:
    if content is None:
        return {}

    if isinstance(content, list):
        content = ''.join(
            part.get('text', '') if isinstance(part, dict) else str(part)
            for part in content
        )

    if not isinstance(content, str):
        return {}

    try:
        payload = json.loads(content)
    except json.JSONDecodeError:
        return {}

    return payload if isinstance(payload, dict) else {}


def github_raw_url(filename: str) -> str:
    encoded_filename = quote(filename)
    return f'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/{encoded_filename}'


def display_input(image_url: str, filename: str) -> None:
    suffix = Path(filename).suffix.lower()
    if suffix in {'.png', '.jpg', '.jpeg', '.webp', '.gif', '.bmp'}:
        display(Image(url=image_url))
    elif suffix == '.pdf':
        print('PDF preview skipped.')


def build_layout_markdown(layout_payload: dict[str, object]) -> str:
    markdown = str(layout_payload.get('md_results') or '').strip()
    if markdown:
        return markdown

    layout_details = layout_payload.get('layout_details') or []
    if not isinstance(layout_details, list):
        return ''

    pages: list[str] = []
    for page_index, page_details in enumerate(layout_details, start=1):
        if not isinstance(page_details, list):
            continue

        parts: list[str] = []
        for detail in page_details:
            if not isinstance(detail, dict):
                continue

            content = str(detail.get('content') or '').strip()
            if content:
                parts.append(content)

        if parts:
            pages.append(f'## Page {page_index}\n\n' + '\n\n'.join(parts))

    return '\n\n'.join(pages).strip()


def price_per_million(model_name: str) -> dict[str, float]:
    price_table = {
        'glm-ocr': {'input': 0.03, 'output': 0.03},
        'glm-5': {'input': 1.0, 'output': 3.2},
    }
    normalized_name = model_name.lower()
    if normalized_name not in price_table:
        raise ValueError(f'No pricing configured for model: {model_name}')

    return price_table[normalized_name]


def process_file(client: ZaiClient, filename: str, image_url: str, layout_model_name: str, chat_model_name: str) -> dict[str, object]:
    layout_pricing = price_per_million(layout_model_name)
    chat_pricing = price_per_million(chat_model_name)

    print(f'Input file: {filename}')
    print(f'Image URL: {image_url}')
    display_input(image_url, filename)

    layout_response = client.layout_parsing.create(
        model=layout_model_name,
        file=image_url,
    )
    layout_payload = layout_response.to_dict(mode='json', exclude_none=True)
    layout_usage = layout_payload.get('usage') or {}
    layout_markdown = build_layout_markdown(layout_payload)
    layout_input_tokens = int(layout_usage.get('prompt_tokens') or 0)
    layout_output_tokens = int(layout_usage.get('completion_tokens') or 0)
    layout_tokens = int(layout_usage.get('total_tokens') or 0)
    if not layout_tokens:
        layout_tokens = layout_input_tokens + layout_output_tokens

    if not layout_markdown:
        data_info = layout_payload.get('data_info') or {}
        raise ValueError(
            'Layout parsing returned no markdown content. ' +
            f"request_id={layout_payload.get('request_id')!r}, " +
            f"num_pages={data_info.get('num_pages')!r}, " +
            f"has_layout_details={bool(layout_payload.get('layout_details'))}"
        )

    extraction_response = client.chat.completions.create(
        model=chat_model_name,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {
                'role': 'user',
                'content': build_extraction_prompt(layout_markdown),
            },
        ],
    )

    usage = extraction_response.usage
    extraction_input_tokens = int(usage.prompt_tokens or 0) if usage else 0
    extraction_output_tokens = int(usage.completion_tokens or 0) if usage else 0
    extraction_tokens = int(usage.total_tokens or 0) if usage else 0
    if not extraction_tokens:
        extraction_tokens = extraction_input_tokens + extraction_output_tokens

    total_input_tokens = layout_input_tokens + extraction_input_tokens
    total_output_tokens = layout_output_tokens + extraction_output_tokens
    total_tokens = layout_tokens + extraction_tokens

    layout_input_cost_usd = layout_input_tokens / 1_000_000 * layout_pricing['input']
    layout_output_cost_usd = layout_output_tokens / 1_000_000 * layout_pricing['output']
    chat_input_cost_usd = extraction_input_tokens / 1_000_000 * chat_pricing['input']
    chat_output_cost_usd = extraction_output_tokens / 1_000_000 * chat_pricing['output']
    input_cost_usd = layout_input_cost_usd + chat_input_cost_usd
    output_cost_usd = layout_output_cost_usd + chat_output_cost_usd
    cost_usd = input_cost_usd + output_cost_usd

    structured_data = parse_json_content(extraction_response.choices[0].message.content)

    raw_total_usages = structured_data.get('total_usages')
    try:
        total_usages = float(str(raw_total_usages).replace(',', '')) if raw_total_usages is not None else None
    except (TypeError, ValueError):
        total_usages = None

    raw_usage_unit = structured_data.get('usage_unit')
    usage_unit = str(raw_usage_unit).strip() if raw_usage_unit not in (None, '') else None

    usage_summary = 'not found'
    if total_usages is not None:
        usage_summary = f'{total_usages:g}'
        if usage_unit:
            usage_summary = f'{usage_summary} {usage_unit}'

    print(f'Layout parsing input tokens: {layout_input_tokens}')
    print(f'Layout parsing output tokens: {layout_output_tokens}')
    print(f'Layout parsing input cost (USD): ${layout_input_cost_usd:.8f}')
    print(f'Layout parsing output cost (USD): ${layout_output_cost_usd:.8f}')
    print(f'Chat completion input tokens: {extraction_input_tokens}')
    print(f'Chat completion output tokens: {extraction_output_tokens}')
    print(f'Chat completion input cost (USD): ${chat_input_cost_usd:.8f}')
    print(f'Chat completion output cost (USD): ${chat_output_cost_usd:.8f}')
    print(f'Total input tokens: {total_input_tokens}')
    print(f'Total output tokens: {total_output_tokens}')
    print(f'Total input cost (USD): ${input_cost_usd:.8f}')
    print(f'Total output cost (USD): ${output_cost_usd:.8f}')
    print(f'Total tokens: {total_tokens}')
    print(f'Total cost (USD): ${cost_usd:.8f}')
    print(f'Total usages: {usage_summary}')
    print()

    return {
        'filename': filename,
        'image_url': image_url,
        'layout_input_tokens': layout_input_tokens,
        'layout_output_tokens': layout_output_tokens,
        'layout_tokens': layout_tokens,
        'chat_input_tokens': extraction_input_tokens,
        'chat_output_tokens': extraction_output_tokens,
        'chat_tokens': extraction_tokens,
        'total_input_tokens': total_input_tokens,
        'total_output_tokens': total_output_tokens,
        'total_tokens': total_tokens,
        'input_cost_usd': input_cost_usd,
        'output_cost_usd': output_cost_usd,
        'cost_usd': cost_usd,
        'total_usages': total_usages,
        'usage_unit': usage_unit,
        'layout_markdown': layout_markdown,
        'structured_data': structured_data,
    }


In [31]:
INPUT_DIR = Path('images/total_usages')
AVAILABLE_INPUTS = [
    {
        'filename': path.name,
        'image_url': github_raw_url(path.name),
    }
    for path in sorted(INPUT_DIR.iterdir())
    if path.is_file()
]

LAYOUT_MODEL_NAME = 'glm-ocr'
CHAT_MODEL_NAME = 'glm-5'

if not AVAILABLE_INPUTS:
    raise ValueError(f'No files found in {INPUT_DIR}')

load_env_file()
api_key = os.getenv('ZAI_API_KEY') or os.getenv('API_KEY')
if not api_key:
    raise ValueError('Set ZAI_API_KEY or API_KEY in the environment or .env before running this notebook.')

client = ZaiClient(api_key=api_key)
AVAILABLE_INPUTS


[{'filename': 'Bill sample breakout.pdf',
  'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Bill%20sample%20breakout.pdf'},
 {'filename': 'Electric_Bill-01.jpg',
  'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Electric_Bill-01.jpg'},
 {'filename': 'cambridge_water.jpg',
  'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/cambridge_water.jpg'},
 {'filename': 'sample-bill.jpg',
  'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/sample-bill.jpg'}]

In [20]:
process_file(client, AVAILABLE_INPUTS[1]['filename'], AVAILABLE_INPUTS[1]['image_url'], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)


Input file: Electric_Bill-01.jpg
Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Electric_Bill-01.jpg


Layout parsing input tokens: 1574
Layout parsing output tokens: 1018
Layout parsing input cost (USD): $0.00004722
Layout parsing output cost (USD): $0.00003054
Chat completion input tokens: 1158
Chat completion output tokens: 468
Chat completion input cost (USD): $0.00115800
Chat completion output cost (USD): $0.00149760
Total input tokens: 2732
Total output tokens: 1486
Total input cost (USD): $0.00120522
Total output cost (USD): $0.00152814
Total tokens: 4218
Total cost (USD): $0.00273336
Total usages: 1000 kWh



{'filename': 'Electric_Bill-01.jpg',
 'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Electric_Bill-01.jpg',
 'layout_input_tokens': 1574,
 'layout_output_tokens': 1018,
 'layout_tokens': 2592,
 'chat_input_tokens': 1158,
 'chat_output_tokens': 468,
 'chat_tokens': 1626,
 'total_input_tokens': 2732,
 'total_output_tokens': 1486,
 'total_tokens': 4218,
 'input_cost_usd': 0.00120522,
 'output_cost_usd': 0.0015281399999999999,
 'cost_usd': 0.0027333599999999998,
 'total_usages': 1000.0,
 'usage_unit': 'kWh',
 'layout_markdown': '![](page=0,bbox=[133, 104, 337, 189])\n\nSetra Systems\n\n159 Swason Road\n\nBoxbourough MA 01719\n\n```markdown\n\n```\n\nPlease Pay By\n\nDec. 31st 2016\n\nPlease Pay Amount $270.00\n\n## Service Provided To:\n\nSetra Systems\n\nAmount Enclosed\n\n159 Swason Road\n\nBoxbourough MA 01719\n\n![](page=0,bbox=[134, 1081, 182, 1137])\n\n## 3 Electric Bill Comparison\n\n<table border="1"><tr><td></td><td>Current Month</t

In [21]:
process_file(client, AVAILABLE_INPUTS[2]['filename'], AVAILABLE_INPUTS[2]['image_url'], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)


Input file: cambridge_water.jpg
Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/cambridge_water.jpg


Layout parsing input tokens: 736
Layout parsing output tokens: 160
Layout parsing input cost (USD): $0.00002208
Layout parsing output cost (USD): $0.00000480
Chat completion input tokens: 388
Chat completion output tokens: 329
Chat completion input cost (USD): $0.00038800
Chat completion output cost (USD): $0.00105280
Total input tokens: 1124
Total output tokens: 489
Total input cost (USD): $0.00041008
Total output cost (USD): $0.00105760
Total tokens: 1613
Total cost (USD): $0.00146768
Total usages: 20 m³



{'filename': 'cambridge_water.jpg',
 'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/cambridge_water.jpg',
 'layout_input_tokens': 736,
 'layout_output_tokens': 160,
 'layout_tokens': 896,
 'chat_input_tokens': 388,
 'chat_output_tokens': 329,
 'chat_tokens': 717,
 'total_input_tokens': 1124,
 'total_output_tokens': 489,
 'total_tokens': 1613,
 'input_cost_usd': 0.00041008,
 'output_cost_usd': 0.0010576,
 'cost_usd': 0.0014676799999999999,
 'total_usages': 20.0,
 'usage_unit': 'm³',
 'layout_markdown': '![](page=0,bbox=[133, 0, 205, 22])\n\n<table class="table table-bordered"><thead><tr><th></th><th>Date</th><th>Reading m³*</th><th>Reading Type</th><th>Volume m³*</th></tr></thead><tbody><tr><td>Present Reading</td><td>2nd January 24</td><td>1069</td><td>Actual Reading</td><td>20</td></tr><tr><td>Previous Reading</td><td>1st July 23</td><td>1049</td><td>Actual Reading</td><td></td></tr></tbody></table>\n\nCharges collected on behalf of Ang

In [22]:
process_file(client, AVAILABLE_INPUTS[3]['filename'], AVAILABLE_INPUTS[3]['image_url'], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)


Input file: sample-bill.jpg
Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/sample-bill.jpg


Layout parsing input tokens: 816
Layout parsing output tokens: 1085
Layout parsing input cost (USD): $0.00002448
Layout parsing output cost (USD): $0.00003255
Chat completion input tokens: 1263
Chat completion output tokens: 391
Chat completion input cost (USD): $0.00126300
Chat completion output cost (USD): $0.00125120
Total input tokens: 2079
Total output tokens: 1476
Total input cost (USD): $0.00128748
Total output cost (USD): $0.00128375
Total tokens: 3555
Total cost (USD): $0.00257123
Total usages: 1414 CF



{'filename': 'sample-bill.jpg',
 'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/sample-bill.jpg',
 'layout_input_tokens': 816,
 'layout_output_tokens': 1085,
 'layout_tokens': 1901,
 'chat_input_tokens': 1263,
 'chat_output_tokens': 391,
 'chat_tokens': 1654,
 'total_input_tokens': 2079,
 'total_output_tokens': 1476,
 'total_tokens': 3555,
 'input_cost_usd': 0.00128748,
 'output_cost_usd': 0.00128375,
 'cost_usd': 0.00257123,
 'total_usages': 1414.0,
 'usage_unit': 'CF',
 'layout_markdown': '![](page=0,bbox=[65, 24, 102, 56])\n\nCustomer Name: JANE AND JOHN DOE\n\n![](page=0,bbox=[374, 50, 391, 65])\n\n$ \\textcircled{2} $ Billing Date: April 13, 2021\n\nFOR CUSTOMER SERVICE\n\n(888) 490-3741\n\nwww.wawater.com\n\n5410 189 Street E\n\nPuyallup, WA 98375\n\n$\\textcircled{1}$ Account Number: 0000000000\n\n![](page=0,bbox=[12, 142, 28, 156])\n\n## 4 Customer Message(s)\n\nVisit www.wawater.com to find out how you can save time, eliminate p

In [32]:
process_file(client, AVAILABLE_INPUTS[0]['filename'], AVAILABLE_INPUTS[0]['image_url'], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)

Input file: Bill sample breakout.pdf
Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Bill%20sample%20breakout.pdf
PDF preview skipped.
Layout parsing input tokens: 1850
Layout parsing output tokens: 889
Layout parsing input cost (USD): $0.00005550
Layout parsing output cost (USD): $0.00002667
Chat completion input tokens: 1110
Chat completion output tokens: 542
Chat completion input cost (USD): $0.00111000
Chat completion output cost (USD): $0.00173440
Total input tokens: 2960
Total output tokens: 1431
Total input cost (USD): $0.00116550
Total output cost (USD): $0.00176107
Total tokens: 4391
Total cost (USD): $0.00292657
Total usages: 700 kWh



{'filename': 'Bill sample breakout.pdf',
 'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/total_usages/Bill%20sample%20breakout.pdf',
 'layout_input_tokens': 1850,
 'layout_output_tokens': 889,
 'layout_tokens': 2739,
 'chat_input_tokens': 1110,
 'chat_output_tokens': 542,
 'chat_tokens': 1652,
 'total_input_tokens': 2960,
 'total_output_tokens': 1431,
 'total_tokens': 4391,
 'input_cost_usd': 0.0011655,
 'output_cost_usd': 0.0017610699999999998,
 'cost_usd': 0.00292657,
 'total_usages': 700.0,
 'usage_unit': 'kWh',
 'layout_markdown': '![](page=0,bbox=[124, 81, 332, 187])\n\n## Your Electricity Statement\n\n[CUSTOMER NAME]\n\nYour account number is:\n\nFor the period of: November 2, 2020 - December 2, 2020\n\n123456123456\n\nThis statement is issued on: December 4, 2020\n\nWhat do I owe?\n\n$121. $^{23}\n\nSee reverse for a summary of how we calculated your charges\n\nHow much did I use?\n\nYou powered your home with\n\n![](page=0,bbox=[606, 464, 719